# Korrektheitsprüfung aller Meldungen (Abschnitt 6.5)

Hier wird das erzeugte angereicherte Bundle mit dem gespeicherten CSAF-Dokument verglichen.
Die erwarteten Werte werden dabei direkt aus dem CSAF-Dokument bestimmt und nicht aus dem Pipeline-Code:


1. CVEs aus den vulnerabilities
2. betroffene Produkte aus known_affected und last_affected
3. Zuordnungen aus den jeweiligen product_status-Einträgen



In [11]:
import csv
import json
from pathlib import Path

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
BUNDLE_DIR = DATA_DIR / "bundles_final"

with open(DATA_DIR / "stats.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f, delimiter=";"))

latest = max(r["run_at"] for r in rows)
MELDUNGEN = sorted({r["wid_id"] for r in rows if r["run_at"] == latest})

print(f"{len(MELDUNGEN)} Meldungen aus dem Lauf {latest}")


250 Meldungen aus dem Lauf 2026-09-24T17:32:36.000Z


In [13]:
def lade_csaf(wid):
    obj = json.loads(
        (RAW_DIR / f"{wid}_csaf.json").read_text(encoding="utf-8")
    )

    # Bei manchen Dateien liegt das Dokument noch eine Ebene tiefer.
    if "document" in obj:
        return obj

    for value in obj.values():
        if isinstance(value, dict) and "document" in value:
            return value

    raise ValueError(f"{wid}: kein CSAF-Dokument gefunden")

def produktnamen(doc):
    namen = {}

    def add_branches(branches):
        for branch in branches or []:
            product = branch.get("product")
            if product:
                pid = product["product_id"]
                namen[pid] = product.get("name", pid)
            add_branches(branch.get("branches"))

    product_tree = doc.get("product_tree", {})
    add_branches(product_tree.get("branches"))

    for product in product_tree.get("full_product_names", []) or []:
        pid = product["product_id"]
        namen[pid] = product.get("name", pid)

    for relation in product_tree.get("relationships", []) or []:
        product = relation.get("full_product_name")
        if product:
            pid = product["product_id"]
            namen[pid] = product.get("name", pid)

    return namen

def soll_aus_csaf(doc):
    namen = produktnamen(doc)

    cves = set()
    betroffen = set()
    behoben = set()
    paare = set()

    for vulnerability in doc.get("vulnerabilities", []) or []:
        cve = vulnerability.get("cve")
        status = vulnerability.get("product_status", {}) or {}

        affected = (status.get("known_affected") or []) + (status.get("last_affected") or [])
        fixed = (status.get("fixed") or []) + (status.get("first_fixed") or [])

        betroffen.update(namen.get(pid, pid) for pid in affected)
        behoben.update(namen.get(pid, pid) for pid in fixed)

        if cve:
            cves.add(cve)
            paare.update((namen.get(pid, pid), cve) for pid in affected)

    return cves, betroffen, behoben, paare

def ist_aus_bundle(wid):
    bundle = json.loads(
        (BUNDLE_DIR / f"{wid}_enriched.json").read_text(encoding="utf-8")
    )
    objs = bundle["objects"]
    by_id = {obj["id"]: obj for obj in objs}

    cves = {obj["name"] for obj in objs if obj["type"] == "vulnerability"}
    software = {obj["name"] for obj in objs if obj["type"] == "software"}

    paare = {
        (by_id[obj["source_ref"]]["name"], by_id[obj["target_ref"]]["name"])
        for obj in objs
        if obj["type"] == "relationship"
        and obj["source_ref"] in by_id
        and obj["target_ref"] in by_id
    }

    anzahl_beziehungen = sum(obj["type"] == "relationship" for obj in objs)

    return cves, software, paare, anzahl_beziehungen

In [14]:
details = []
rueckfall = []
fehlend = []

summe = {"cves": 0, "produkte": 0, "beziehungen": 0}
ok = {"cves": 0, "produkte": 0, "beziehungen": 0}

for wid in MELDUNGEN:
    try:
        doc = lade_csaf(wid)
        i_cve, i_sw, i_paar, n_rel = ist_aus_bundle(wid)
    except (FileNotFoundError, ValueError) as e:
        fehlend.append((wid, str(e)))
        continue

    s_cve, s_aff, s_fix, s_paar = soll_aus_csaf(doc)

    ok_cve = s_cve == i_cve
    ok_rel = i_paar == s_paar and n_rel == len(i_paar)

    summe["cves"] += len(s_cve)
    summe["beziehungen"] += len(s_paar)
    ok["cves"] += ok_cve
    ok["beziehungen"] += ok_rel

    # Wenn im CSAF keine betroffenen Produkte angegeben sind,
    # greift die Pipeline auf die Kategorie im Produktbaum zurück.
    if not s_aff:
        rueckfall.append(wid)
        ok_prod = True
    else:
        falsch_eingeordnet = (i_sw & s_fix) - s_aff
        ok_prod = i_sw == s_aff and not falsch_eingeordnet

        summe["produkte"] += len(s_aff)
        ok["produkte"] += ok_prod

    if not (ok_cve and ok_prod and ok_rel):
        details.append((wid, {
            "CVE fehlt im Bundle": sorted(s_cve - i_cve),
            "CVE zusaetzlich im Bundle": sorted(i_cve - s_cve),
            "Produkt fehlt im Bundle": sorted(s_aff - i_sw) if s_aff else [],
            "Produkt zusaetzlich im Bundle": sorted(i_sw - s_aff) if s_aff else [],
            "Zuordnung fehlt": sorted(s_paar - i_paar)[:10],
            "Zuordnung zusaetzlich": sorted(i_paar - s_paar)[:10],
            "doppelte Beziehungsobjekte": n_rel - len(i_paar)
        }))

n = len(MELDUNGEN) - len(fehlend)

print(f"Gepruefte Meldungen: {n}")
print(f"  CVEs korrekt:         {ok['cves']} von {n}")
print(f"  Produkte korrekt:     {ok['produkte']} von {n - len(rueckfall)}  (+{len(rueckfall)} Meldungen mit Rueckfall, nicht geprueft)")
print(f"  Beziehungen korrekt:  {ok['beziehungen']} von {n}")
print(f"Gepruefte Einzelangaben: {summe['cves']} CVEs, {summe['produkte']} Produkte, {summe['beziehungen']} Zuordnungen")
print(f"Meldungen mit Abweichung: {len(details)}")

if rueckfall:
    print(f"Rueckfall-Meldungen: {rueckfall}")

if fehlend:
    print(f"Nicht pruefbar (Datei fehlt): {fehlend}")

for wid, ergebnis in details[:20]:
    print(f"\n--- {wid} ---")
    for name, werte in ergebnis.items():
        if werte:
            print(f"  {name}: {werte}")

Gepruefte Meldungen: 250
  CVEs korrekt:         250 von 250
  Produkte korrekt:     250 von 250  (+0 Meldungen mit Rueckfall, nicht geprueft)
  Beziehungen korrekt:  250 von 250
Gepruefte Einzelangaben: 8344 CVEs, 2366 Produkte, 101915 Zuordnungen
Meldungen mit Abweichung: 0
